In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("superstore.csv", encoding="latin1")
print(df.shape)

(9994, 24)


In [4]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)
print(df.columns.tolist())

['row_id', 'order_id', 'order_date', 'year', 'month', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit', 'profit_margin']


In [5]:
df["discount"] = (
    df["discount"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.strip()
    .astype(float)
)

# Normalize to decimal if values are like 20, 45 (not 0.2, 0.45)
if df["discount"].max() > 1:
    df["discount"] = df["discount"] / 100

print(df["discount"].describe())

count    9994.000000
mean        0.156203
std         0.206452
min         0.000000
25%         0.000000
50%         0.200000
75%         0.200000
max         0.800000
Name: discount, dtype: float64


In [6]:
df["profit_margin"] = (
    df["profit_margin"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.strip()
    .astype(float)
)

if df["profit_margin"].max() > 1:
    df["profit_margin"] = df["profit_margin"] / 100

print(df["profit_margin"].describe())

count    9994.000000
mean        0.120572
std         0.466918
min        -2.750000
25%         0.080000
50%         0.270000
75%         0.360000
max         0.500000
Name: profit_margin, dtype: float64


In [7]:
df["sales"] = pd.to_numeric(df["sales"], errors="coerce")
df["profit"] = pd.to_numeric(df["profit"], errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")

print(df[["sales", "profit", "quantity"]].dtypes)

sales       float64
profit      float64
quantity      int64
dtype: object


In [8]:
df = df.drop(columns=["row_id"])  # auto-increment will handle this in DB
print(df.shape)

(9994, 23)


In [9]:
# Cell 9 — Check for nulls after cleaning
print(df.isnull().sum())

order_id         0
order_date       0
year             0
month            0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0
category         0
sub_category     0
product_name     0
sales            0
quantity         0
discount         0
profit           0
profit_margin    0
dtype: int64


In [10]:
# Cell 10 — Final Sanity Check
print(df.dtypes)
print("\n")
df.head(3)

order_id             str
order_date           str
year               int64
month                str
ship_date            str
ship_mode            str
customer_id          str
customer_name        str
segment              str
country              str
city                 str
state                str
postal_code        int64
region               str
product_id           str
category             str
sub_category         str
product_name         str
sales            float64
quantity           int64
discount         float64
profit           float64
profit_margin    float64
dtype: object




,order_id,order_date,year,month,ship_date,ship_mode,customer_id,customer_name,segment,country,...,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,profit_margin
0,CA-2016-152156,08/11/2016,2016,November,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,...,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136,0.16
1,CA-2016-152156,08/11/2016,2016,November,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,...,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820,0.30
2,CA-2016-138688,12/06/2016,2016,June,16/06/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,...,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714,0.47


In [12]:
df["order_date"] = pd.to_datetime(df["order_date"], format="%d/%m/%Y")
df["ship_date"] = pd.to_datetime(df["ship_date"], format="%d/%m/%Y")

# Verify
print(df["order_date"].dtype)
print(df["ship_date"].dtype)
print(df["order_date"].head())

datetime64[us]
datetime64[us]
0   2016-11-08
1   2016-11-08
2   2016-06-12
3   2015-10-11
4   2015-10-11
Name: order_date, dtype: datetime64[us]


In [13]:

df["order_date"].dt.month        # gives 1-12
df["order_date"].dt.month_name() # gives "January", "February" etc.
df["order_date"].dt.year         # gives 2014, 2015 etc.

0       2016
1       2016
2       2016
3       2015
4       2015
        ... 
9989    2014
9990    2017
9991    2017
9992    2017
9993    2017
Name: order_date, Length: 9994, dtype: int32

In [14]:
df.to_csv("superstore_cleaned.csv", index=False)
print("Saved successfully!")

Saved successfully!


In [15]:
df = pd.read_csv("superstore_cleaned.csv")
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"] = pd.to_datetime(df["ship_date"])
print(df.shape)
print(df.dtypes)

(9994, 23)
order_id                    str
order_date       datetime64[us]
year                      int64
month                       str
ship_date        datetime64[us]
ship_mode                   str
customer_id                 str
customer_name               str
segment                     str
country                     str
city                        str
state                       str
postal_code               int64
region                      str
product_id                  str
category                    str
sub_category                str
product_name                str
sales                   float64
quantity                  int64
discount                float64
profit                  float64
profit_margin           float64
dtype: object


In [16]:
# Fix postal_code to string
df["postal_code"] = df["postal_code"].astype(str)

# Verify
print(df["postal_code"].dtype)
print(df["postal_code"].head())

str
0    42420
1    42420
2    90036
3    33311
4    33311
Name: postal_code, dtype: str


In [17]:
df.to_csv("superstore_cleaned.csv", index=False)
print("Saved!")

Saved!
